In [1]:

import pandas as pd

# df = pd.read_csv("./data/training_dataset_id50_unique_nolabel2_v4_CST.csv")
df = pd.read_csv("./data/v8_generative_output.csv")

In [2]:
df

,TMC_seq,Template
0,FGPDGRLKTWVYGVAAGAFVLLIFIVSMIYLACKKPKKPQRRQNNR...,THSD7A_original
1,__PDGRLKT__YGVAAGAF______VSM____CKKPKKPQRRQ_NR...,THSD7A_prompt
2,RSPDGRLKTVDYGVAAGAFLTVAFAVSMAHAGCKKPKKPQRRQFNR...,THSD7A
3,GHPDGRLKTWVYGVAAGAFFPLFFLVSMFLPYCKKPKKPQRRQLNR...,THSD7A
4,GDPDGRLKTISYGVAAGAFMLLLISVSMNYYCCKKPKKPQRRQENR...,THSD7A
...,...,...
1117,GLKAGVIAVAALVGIAVVAGIRATISSRKKRMAKYEKAEIKEEDKE...,EPCAM_v3
1118,MLKAGVIAVYIFIAIAVVAGIWFTVKSRKKRMAKYEKAEIKEEDAA...,EPCAM_v3
1119,GLKAGVIAVVIIPTIAVVAGISAIFVSRKKRMAKYEKAEIKEVAID...,EPCAM_v3
1120,GLKAGVIAVVVFGLIAVVAGIFFAIFSRKKRMAKYEKAEIKEADLA...,EPCAM_v3


In [4]:


sequences = df["TMC_seq"].tolist()
# labels = df["CST_category"].tolist()
# accessions = df["Accession"].tolist()


In [5]:
import torch
import torch.distributed as dist
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model_checkpoint = "./esm2_t12_35M_category_CST_homolog_v6/checkpoint-10194"

num_labels = 4  # Add 1 since 0 can be a label

# load model from local directory, esm2_t12_35M_UR50D-SPTM_CM_35M_5epochs_split5

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)
# model = AutoModelForSequenceClassification.from_pretrained("./esm2_t12_35M_UR50D-SPTM_CM_35M_5epochs_split5/checkpoint-1996", num_labels=num_labels)
# model = AutoModelForSequenceClassification.from_pretrained("./esm2_t12_35M_UR50D-SPTM_CM_35M_split5epoch2_split20_3epochs/checkpoint-998", num_labels=num_labels)
# model = AutoModelForSequenceClassification.from_pretrained("./esm2_t30_150M_UR50D-SPTM_CM_150M_5epochs/checkpoint-1682", num_labels=num_labels)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
trainer = Trainer(model=model, tokenizer=tokenizer)

print("Let's use", torch.cuda.device_count(), "GPUs!")

#  model = nn.DataParallel(model)

Let's use 1 GPUs!


In [8]:

input_sequence = sequences
len(input_sequence)

1122

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

inputs = tokenizer(input_sequence, return_tensors="pt", padding=True, truncation=True).to(device)
model.to(device)


# Make predictions

batch_size = 4
# Function to process in batches
def process_in_batches(inputs, model, batch_size):
    all_outputs = []
    for i in range(0, len(inputs["input_ids"]), batch_size):
        # Prepare the batch inputs
        batch = {k: v[i:i + batch_size].to(device) for k, v in inputs.items()}

        # Perform inference
        with torch.no_grad():
            outputs = model(**batch)
            all_outputs.append(outputs.logits)

    # Concatenate all the outputs
    return torch.cat(all_outputs)

# Run the batched inference
outputs = process_in_batches(inputs, model, batch_size)

In [10]:

predictions = torch.argmax(outputs, axis=1)
print("Predicted class:", predictions)

Predicted class: tensor([3, 0, 2,  ..., 3, 3, 3], device='cuda:0')


In [12]:
df

,TMC_seq,Template,predicted class
0,FGPDGRLKTWVYGVAAGAFVLLIFIVSMIYLACKKPKKPQRRQNNR...,THSD7A_original,3
1,__PDGRLKT__YGVAAGAF______VSM____CKKPKKPQRRQ_NR...,THSD7A_prompt,0
2,RSPDGRLKTVDYGVAAGAFLTVAFAVSMAHAGCKKPKKPQRRQFNR...,THSD7A,2
3,GHPDGRLKTWVYGVAAGAFFPLFFLVSMFLPYCKKPKKPQRRQLNR...,THSD7A,2
4,GDPDGRLKTISYGVAAGAFMLLLISVSMNYYCCKKPKKPQRRQENR...,THSD7A,3
...,...,...,...
1117,GLKAGVIAVAALVGIAVVAGIRATISSRKKRMAKYEKAEIKEEDKE...,EPCAM_v3,3
1118,MLKAGVIAVYIFIAIAVVAGIWFTVKSRKKRMAKYEKAEIKEEDAA...,EPCAM_v3,0
1119,GLKAGVIAVVIIPTIAVVAGISAIFVSRKKRMAKYEKAEIKEVAID...,EPCAM_v3,3
1120,GLKAGVIAVVVFGLIAVVAGIFFAIFSRKKRMAKYEKAEIKEADLA...,EPCAM_v3,3


In [11]:

df["predicted class"] = predictions.tolist()
df.to_csv("v8_generative_output_predicted.csv")